# AOS Pipeline Runner

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** In Progress  
**Keywords:** AOS, pipeline, mktable, DZ fitting, plots  

## Description

Notebook interface for the `run_pipeline.py` pipeline tracker.
Reads `param_sets.yaml` and `runs.yaml`, executes steps, and updates status.

Key functionality:
1. Show status of all pipeline runs
2. Execute pending steps: mktable -> fit -> plots
3. Capture per-step output to `output/{run}_{step}.log`

**Output:** HDF5 donut+visits tables, fit parquet files, plot PDFs, per-step logs

**Based on:** `code/run_pipeline.py` CLI, `param_sets.yaml`, `runs.yaml`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |
| 2026-04-20 | Aaron Roodman | Added chunk4 using new `fam_danish_v1_triplets_bin1x` param_set (Danish v1.0 triplets bin 1x, 2026-03-15 through 2026-04-09, single chunk covering all available dates) |
| 2026-04-20 | Aaron Roodman | Added chunk5 using new `fam_danish_v1_triplets` param_set (Danish v1.0 triplets, bin=2, same date range as chunk4); bin 1x collection not yet available |
| 2026-04-20 | Aaron Roodman | Switched output format from single HDF5 to per-visit row-group parquet (donut table) + sidecar visits parquet. Streaming I/O both ways. List columns are natively parquet-typed, so the donut file is readable by any parquet-aware tool. Legacy HDF5 files are still readable for re-fitting chunks 1-3. Added detailed logging of missing ConsDB rotator angles before falling back to Butler visitInfo + EFD. |
| 2026-04-21 | Aaron Roodman | Added `movie` / `movie_ccs` pipeline steps to handle the slow per-image residual maps + ffmpeg separately from `plots`.  `plots` now skips single-image+movie by default.  Also: `step_name` accepts a list (e.g. `['fit', 'plots']`) to run several steps in one go. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Status](#status)
4. [Run Steps](#run)
5. [Manual Status Updates](#manual)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Which run and step(s) to execute.
#   step_name = None              -> all pending steps
#   step_name = 'fit'             -> single step
#   step_name = ['fit', 'plots']  -> several steps in one go
#
# Available steps:
#   mktable, fit, plots, movie, fit_ccs, plots_ccs, movie_ccs
#
# `plots` produces fit-parameter PDFs and trio comparisons only.
# `movie` produces the per-image residual maps + ffmpeg movie (slow).
run_name = 'chunk1'
step_name = ['fit', 'plots']
dry_run = False

# Set True to overwrite existing mktable output parquet files (otherwise
# run_mktable refuses to clobber them and raises FileExistsError).
OVERWRITE = False

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import io
import contextlib
import subprocess
import sys
import time
from datetime import datetime

sys.path.insert(0, 'code')
from run_pipeline import (
    load_runs, save_runs, load_param_sets, resolve_run,
    hdf5_path, fits_path, fits_path_ccs, plots_dir_ccs, step_log_path,
    build_command,
    cmd_status, cmd_run, cmd_set, cmd_reset,
    git_sync,
    STEP_ORDER, STEP_DEPS, LOG_DIR, LOG_FILE, log,
)
from intrinsics_lib import run_mktable
from dz_fitting import run_double_zernike_fits

print('Pipeline tools loaded.')

<a id='status'></a>
## 3. Status

In [24]:
data = load_runs()
cmd_status(data)


Run                  param_set                 days                   coord   mktable      fit    plots
----------------------------------------------------------------------------------------------------
chunk1               fam_danish_triplets       20260315-20260317    OCS      done     done      ---
chunk2               fam_danish_triplets       20260324-20260327    OCS       ---      ---      ---
chunk3               fam_danish_triplets       20260331-20260404    OCS       ---      ---      ---

0/3 runs fully complete.



<a id='run'></a>
## 4. Run Steps

Executes pending pipeline steps based on the parameters cell above.

- `run_name = None` → all runs
- `step_name = None` → all pending steps (respecting prerequisites)
- `dry_run = True` → show commands without executing

Output is captured to `output/{run}_{step}.log` and also printed here.

In [ ]:
data = load_runs()
param_sets = load_param_sets()

# Normalize step_name to a list (or None)
if step_name is None:
    step_filter = None
elif isinstance(step_name, str):
    step_filter = [step_name]
else:
    step_filter = list(step_name)

if dry_run:
    cmd_run(data, run_name, step_filter, dry_run=True)
else:
    runs = data.get('runs', {})
    targets = {run_name: runs[run_name]} if run_name else runs

    for name, cfg in targets.items():
        resolved = resolve_run(cfg, param_sets)
        steps = cfg.get('steps', {})

        for step in STEP_ORDER:
            if step_filter is not None and step not in step_filter:
                continue
            status = steps.get(step, 'pending')
            if status not in ('pending', 'failed'):
                continue

            # Prerequisite check — treat batch-mate steps as effectively
            # satisfied (we just ran or will run them).
            def _prereq_ok(s):
                return (steps.get(s, 'pending') == 'done'
                        or (step_filter is not None and s in step_filter))
            prereqs_ok = all(_prereq_ok(s) for s in STEP_DEPS.get(step, []))
            if not prereqs_ok:
                print(f'{name}.{step}: prerequisites not met, skipping')
                continue

            print(f'\n{"="*60}')
            print(f'{name}.{step}: STARTING')
            print(f'{"="*60}')

            data['runs'][name]['steps'][step] = 'running'
            save_runs(data)

            step_logfile = step_log_path(name, step)
            LOG_DIR.mkdir(parents=True, exist_ok=True)
            t0 = time.time()
            step_success = False

            try:
                if step == 'mktable':
                    mktable_params = dict(resolved)
                    for k in ['param_set', 'no_single_image',
                              'no_fit_params', 'no_trio']:
                        mktable_params.pop(k, None)
                    mktable_params.setdefault('output_dir', 'output')
                    mktable_params.setdefault('include_thermal', True)
                    mktable_params.setdefault('calc_intrinsics', True)
                    mktable_params['overwrite'] = (
                        OVERWRITE or mktable_params.get('overwrite', False))

                    capture = io.StringIO()
                    with contextlib.redirect_stdout(capture):
                        result = await run_mktable(**mktable_params)
                    output_text = capture.getvalue()
                    print(output_text)
                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
                        f.write(output_text)

                elif step in ('fit', 'fit_ccs'):
                    h5 = hdf5_path(resolved)
                    if step == 'fit_ccs':
                        coord = 'CCS'
                        output_file = fits_path_ccs(resolved)
                    else:
                        coord = resolved.get('coord_sys', 'OCS')
                        output_file = None

                    capture = io.StringIO()
                    with contextlib.redirect_stdout(capture):
                        run_double_zernike_fits(
                            input_file=h5, coord_sys=coord,
                            output_file=output_file)
                    output_text = capture.getvalue()
                    print(output_text)
                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
                        f.write(output_text)

                elif step in ('plots', 'plots_ccs', 'movie', 'movie_ccs'):
                    # Build the exact command run_pipeline.py would use,
                    # then run it as a subprocess so the notebook stays
                    # in lockstep with the CLI.
                    cmd = build_command(name, step, resolved)
                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
                        f.write(f'# {" ".join(cmd)}\n\n')
                        f.flush()
                        proc = subprocess.Popen(
                            cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
                        for line in proc.stdout:
                            sys.stdout.write(line)
                            f.write(line)
                        proc.wait()
                    if proc.returncode != 0:
                        raise RuntimeError(f'{step} exited with {proc.returncode}')

                elapsed = time.time() - t0
                data['runs'][name]['steps'][step] = 'done'
                save_runs(data)
                log(f'{name}.{step}: DONE ({elapsed:.0f}s) — log: {step_logfile}')
                step_success = True

            except Exception as e:
                elapsed = time.time() - t0
                data['runs'][name]['steps'][step] = 'failed'
                save_runs(data)
                log(f'{name}.{step}: FAILED ({elapsed:.0f}s) — {e}')
                print(f'Log: {step_logfile}')

            outcome = 'done' if step_success else 'failed'
            git_sync(f'{name}.{step}: {outcome} ({elapsed:.0f}s)')

<a id='manual'></a>
## 5. Manual Status Updates

Use these cells to manually set or reset step status.

In [19]:
# Manually mark a step as done (or pending, failed, skip)
# cmd_set(load_runs(), 'chunk1', 'mktable', 'done')

In [20]:
# Reset all steps for a run
# cmd_reset(load_runs(), 'chunk1')

In [21]:
# Refresh status after changes
data = load_runs()
cmd_status(data)


Run                  param_set                 days                   coord   mktable      fit    plots
----------------------------------------------------------------------------------------------------
chunk1               fam_danish_triplets       20260315-20260317    OCS      done     done      ---
chunk2               fam_danish_triplets       20260324-20260327    OCS       ---      ---      ---
chunk3               fam_danish_triplets       20260331-20260404    OCS       ---      ---      ---

0/3 runs fully complete.

